# [DS4Bio] Red Teaming: Adversarial testing and evaluation of AI-assisted coding
### Data Science for Biology
**Notebook developed by:** *Jeremy Marcus*<br>
**Supervised by:** *Steven E. Brenner*


Last week, we used generative AI to produce code to solve a specific task and wrote code for testing the functionality of that code. This week, we will be performing adversarial testing of each other's code, known as "red teaming". Through this process, we will identify ways to write better program specs, create better tests, and more effectively utilize AI tools.

***

#### **GROUP PROJECT**

This project will be completed collaboratively with your group and submitted together. All members should contribute for each part of the lab.

Below, please list your group number and the names of the team members:

Group Number: 4 - Amoeba Sisters

Group Members:

* Sophie Feng
* Monica Tsai
* Malaika Nazir
* Mayra Alzate
* Ruijie Feng
* Humayd Zameer

#### **Exporting Chats**

When AI tools are used to assist in solving a problem, we will ask for transcripts of any chat sessions as a PDF. Unfortunately, getting Gemini to print a PDF of a chat session while using a CalNet account is not a trivial operation. Here is our suggested solution:

1. Click the `Tools` button at the bottom of the chat box.
2. Select `Canvas`.
3. Prompt Gemini with the following:

> Please transcribe our entire conversation word-for-word into a new Canvas document

4. Print to PDF

Alternatively, there are browser-specific extensions that enable chat export. If you choose to use one of these options ***be sure that the extension is one which runs locally, not uploading any data to the cloud***.

#### Use of AI

Throughout this and future labs, we will be explicit about where use of AI is permitted. There will be three possible options, which will be stated up front for each question:

* <font color = #0fd12c>**AI REQUIRED**</font>
* <font color = #0fd12c>**AI PERMITTED**</font>
* <font color = #0fd12c>**NO AI**</font>

#### Submission requirements:

The following are what you are expected to submit:

* Copy of the `.ipynb` notebook file
* PDF version of the notebook file
* PDF export of Gemini chat logs (if Gemini was used)

***
## Introduction

#### **Red-teaming**

As a reminder from last week, [red-teaming](https://en.wikipedia.org/wiki/Red_team) is a cybersecurity exercise where a group poses as an adversary, attempts to disrupt the functionality of some system, then reports to the owners of that system how it might be improved.

Note that here, "adversarial" is a technical term from the discipline of Security (both for physical and computer systems). In a red-team process, the teams are actively working **together** to improve the end product, not against each other.

Last week, you wrote a suite of tests for the functionality of the function `translate_from_template`. This week, we will have you run another group's version of `translate_from_template` through your tests, evaluate the performance of the function, compile a report on your findings, and discuss the results with the authors of the code. All groups will therefore be acting as both the testers and the tested.

In [1]:
# RUN THIS CELL
from group_solutions import *

***
### 1. Testing


#### **Group Assignments**

Your group will evaluate the code of the group next to you, moving clockwise in a circle around the classroom. The following diagram shows the full chain (arrows mean "pass `translate_from_template` to the next group"):

```
1 → 2 → 3 → 4 → 5 → 6 → 7 → 9
↑                           ↓
←←←←←←←←←←←←←←←←←←←←←←←←←←←←←
```

In the following cell, replace `GROUP_METHOD` with the function name `group_N`, where `N` is the number of the group you are evaluating. For example, Group 1, who is evaluating Group 2, will call the function `group_2`.

In [2]:
def translate_from_template(template_sequence):
    return group_5(template_sequence)

<font color = #d14d0f>**QUESTION 1a**:</font> <font color = #0fd12c>**NO AI**:</font>

Copy your definition of `translation_tester` from last week's lab into the following code block, then run it.

In [6]:
def translation_tester():
    # Store all of your error messages and a count of how many tests you have run
    error_messages = []
    n_tests = 0
    passed_tests = 0

    # Test case 1: loop through some set of inputs that are known not to work for a given reason
    error_inputs = [3, 3.2]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            # Construct a helpful message containing the offending input, and add it to the list of error messages
            error_messages.append(f"Numbers in the input: {template}")
        else:
            passed_tests += 1
            #Test Case: If it's a string
    #Test 2: Empty Input
    error_inputs = ["", "   ", "\n"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            error_messages.append(
                f"Empty input: {template}"
            )
        else:
            passed_tests += 1

    #Test Case 3: Less than 3 nucleotides
    error_inputs = ["A", "CG"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            # Construct a helpful message containing the offending input, and add it to the list of error messages
            error_messages.append(f"Less than 3 nucleotides: {template}")
        else:
            passed_tests += 1

    # Test case 4: for valid nucleotide  
    error_inputs = ["AUHBW"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
           error_mesages.append(f"Template is not correct MRNA nucleotides: A,U,C,G {template}")
        else:
            passed_tests += 1


    #Test Case 5: No Start Codon
    error_inputs = ["CCGTAGTTCGCAAGCCGATATTA"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            error_messages.append(f"No start codon: {template}")
        else:
            passed_tests += 1

    #Test case 6: valid input
    # "TACATT", "M"
    # taccgaatt test lowercase
    # "TACATTCGA" -> AUG UAA GCU expected: "M" (it should stop at UAA and ignore later codons)
    valid_inputs = [("TACATT", "M"), ("taccgaatt", "MA"), ("TACATTCGA", "M")]
    for template, expected in valid_inputs:
        n_tests += 1
        result = translate_from_template(template)

        if result == "ERROR":
            error_messages.append(f"Valid input returned ERROR: {template!r}, expected {expected!r}")
        elif result != expected:
            error_messages.append(f"Wrong amino acid output: {template!r}, expected {expected!r}, got {result!r}")
        else:
            passed_tests += 1




    # Print report
    print(f"Tests passed: {passed_tests}/{n_tests}")
    for message in error_messages:
        print(message)

In [4]:
translation_tester()

Tests passed: 9/12
Wrong amino acid output: 'TACATT', expected 'M', got ''
Wrong amino acid output: 'taccgaatt', expected 'MA', got 'A'
Wrong amino acid output: 'TACATTCGA', expected 'M', got ''


<font color = #d14d0f>**QUESTION 1b**:</font> <font color = #0fd12c>**NO AI**:</font>

The implementation of your assigned code can be found in [group_solutions.py](group_solutions.py). With the context of your error report, take a look at that code and see if you can diagnose the source of any of the errors. This will become part of your response for a later question.

Acknowledge below that you have taken a look and thought through what you found.

We found one error, which seems like a lack of printing M in their sequence. However, it turns out that their code reverses once, feeding a coding strand and treating it as a template. It instead ought to reverse twice, or not at all.

Software errors can be loosely categorized into the following types:

* **Logic Errors**: mistakes in logic lead to unexpected results when the software is run to completion
* **Input Handling Errors**: software does not cleanly handle unexpected or out-of-spec inputs
* **Syntax Errors**: incorrect usage of the language prevents the software from even running
* **Runtime Errors**: other types of errors pop up while the software is running, possibly caused by the computer system itself

Because we know that every group turned in functional code, and the scope of this code is small, you will mostly be running into **Logic** and **Input Handling** errors for this lab.

Additionally, software errors are often categorized by severity or complexity. Failure to handle an unexpected input might only affect a small number of times the function is run, and adding an extra input check is usually a simple task. If the software is routinely outputting incorrect answers, there might be a larger and more fundamental issue at play.

Throughout this process, be thoughtful about the tests you wrote and consider whether they are correct. Did the test fail because `translate_from_template` is incorrect, or because you wrote a bad test?

<font color = #d14d0f>**QUESTION 1c**:</font> <font color = #0fd12c>**NO AI**:</font>

While testing or reading your assigned code, you may determine that you are missing important tests or have written incorrect tests. If this is the case, make updates to your testing method below. Additionally, describe the mistakes you made and how they were addressed.

In [7]:
def translation_tester():
    # Store all of your error messages and a count of how many tests you have run
    error_messages = []
    n_tests = 0
    passed_tests = 0

    # Test case 1: loop through some set of inputs that are known not to work for a given reason
    error_inputs = [3, 3.2]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            # Construct a helpful message containing the offending input, and add it to the list of error messages
            error_messages.append(f"Numbers in the input: {template}")
        else:
            passed_tests += 1
            #Test Case: If it's a string
    #Test 2: Empty Input
    error_inputs = ["", "   ", "\n"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            error_messages.append(
                f"Empty input: {template}"
            )
        else:
            passed_tests += 1

    #Test Case 3: Less than 3 nucleotides
    error_inputs = ["A", "CG"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            # Construct a helpful message containing the offending input, and add it to the list of error messages
            error_messages.append(f"Less than 3 nucleotides: {template}")
        else:
            passed_tests += 1

    # Test case 4: for valid nucleotide  
    error_inputs = ["AUHBW"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
           error_mesages.append(f"Template is not correct MRNA nucleotides: A,U,C,G {template}")
        else:
            passed_tests += 1


    #Test Case 5: No Start Codon
    error_inputs = ["CCGTAGTTCGCAAGCCGATATTA"]
    for template in error_inputs:
        n_tests += 1
        if translate_from_template(template) != "ERROR":
            error_messages.append(f"No start codon: {template}")
        else:
            passed_tests += 1

    #Test case 6: valid input
    # "TACATT", "M"
    # taccgaatt test lowercase
    # "TACATTCGA" -> AUG UAA GCU expected: "M" (it should stop at UAA and ignore later codons)
    valid_inputs = [("TACATT", "M"), ("taccgaatt", "MA"), ("TACATTCGA", "M")]
    for template, expected in valid_inputs:
        n_tests += 1
        result = translate_from_template(template)

        if result == "ERROR":
            error_messages.append(f"Valid input returned ERROR: {template!r}, expected {expected!r}")
        elif result != expected:
            error_messages.append(f"Wrong amino acid output: {template!r}, expected {expected!r}, got {result!r}")
        else:
            passed_tests += 1




    # Print report
    print(f"Tests passed: {passed_tests}/{n_tests}")
    for message in error_messages:
        print(message)

We added an additional test to feed an mrna input instead of a dna input to test whether their code accounts for Uracil in lieu of Thymine

<font color = #d14d0f>**QUESTION 1d**:</font> <font color = #0fd12c>**NO AI**:</font>

Based on your interpretation of the test results above, compile a small report describing the errors you observed. Which test cases failed, and why? Be clear about your findings and provide helpful suggestions for what might need to be fixed.

Consider breaking down the errors by category and severity, and devote more time to any complex logic errors you run into.

The only major error is the reversal step, which was eliminated, as it's technically unecessary.

***
### 2. Reporting and discussing


Once your group has finished **Part 1**, have a discussion with the group whose code you evaluated. Sit down at the same table and walk through your findings.

Note: While this is a fundamentally "adversarial" testing process, these discussions should be constructive and supportive. The goal is to work together to figure out how to improve the code.

As a reminder, if it is helpful to see your group's code, the implementation can be found in [group_solutions.py](group_solutions.py).

<font color = #d14d0f>**QUESTION 2**:</font> <font color = #0fd12c>**NO AI**:</font>

Make notes about the findings from discussing your group's code with the group that tested it. What changes do you plan to make?

The group that tested our code didn't find any errors, but we had also made the same reversal error

<font color = #d14d0f>**QUESTION 3**:</font> <font color = #0fd12c>**NO AI**:</font>

Acknowledge that you have spoken with the group whose code you evaluated.

Acknowledged

***
### 3. OPTIONAL: Addressing


<font color = #d14d0f>**OPTIONAL**:</font> <font color = #0fd12c>**AI PERMITTED**:</font>

Based on your findings above, implement the necessary changes. You may either implement them yourself or work with Gemini to implement them.

We found the same error in our code as the group whose code we evaluated, and changed our code as well